# Flair Feature-Family Target Multitask Training

Train a Flair native multitask model from Quant Warehouse FMP feature families and mirrored event-pair targets.

Shape:

- Universe: FMP common equities with `market_cap >= 1_000_000_000_000`.
- Features: one text row per `(date, symbol, feature_family, target_task)`.
- Text: key-value pairs for feature values only, with numeric values rounded to two decimals. `symbol` and `date` are included only in the explicit `fmp_equity_profile` context feature family.
- Targets: mirrored event-pair tasks such as congress buy vs sell, insider buy vs sell, analyst upgrade vs downgrade, price target raise vs cut, earnings beat vs miss, plus one oracle buy vs sell side task across all configured k values. Non-event dates are excluded.

This notebook intentionally does not run anything at import time beyond configuration. Execute cells manually when you are ready to train.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from time import perf_counter
from datetime import date, datetime
import math
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

WAREHOUSE_REPO = REPO_ROOT.parent / "quant-warehouse"
if WAREHOUSE_REPO.exists() and str(WAREHOUSE_REPO) not in sys.path:
    sys.path.insert(0, str(WAREHOUSE_REPO))

warnings.filterwarnings("ignore", category=UserWarning)

import torch
import flair
from flair.data import Corpus, Dictionary, FlairDataset, Sentence
from flair.embeddings import TransformerDocumentEmbeddings
from flair.models import MultitaskModel, TextClassifier
from flair.trainers import ModelTrainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EVENT_PAIR_TAXONOMY, EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    EventFeatureDatasetConfig,
    FMP_EVENT_CONTEXT_FEATURE_FAMILIES,
    FMP_EQUITY_PROFILE_FEATURE_COLUMNS,
    FMP_EQUITY_PROFILE_FEATURE_FAMILY,
    FMP_EQUITY_PROFILE_FEATURE_SOURCE,
    FamilyEvaluationConfig,
    add_fmp_equity_profile_feature_family,
    add_fmp_event_context_feature_families,
    build_event_feature_text_dataset,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    event_pair_task_specs,
    fmp_event_context_allowed_feature_families_by_task,
    load_fmp_event_pairs,
    oracle_side_task_specs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

In [2]:
RANDOM_SEED = 20260630
MIN_MARKET_CAP = 10_000_000_000
START_DATE = "1990-01-01"
END_DATE = None
TRAIN_END = pd.Timestamp("2023-12-31")
DEV_END = pd.Timestamp("2024-12-31")

MIN_ROWS_PER_TASK = 120
MIN_POSITIVE_ROWS_PER_TASK = 10
MIN_FEATURE_COVERAGE = 0.50
SMOKE_TEST = False
SMOKE_SYMBOL_LIMIT = 2
SMOKE_TARGET_TASK_LIMIT = 3
SMOKE_FEATURE_FAMILY_LIMIT = 5
MAX_ROWS_PER_TASK_SPLIT = None

FLAIR_TRANSFORMER = "prajjwal1/bert-tiny"
FLAIR_EPOCHS = 1
FLAIR_BATCH_SIZE = 256
FLAIR_BATCH_CHUNK_SIZE = 8
FLAIR_EVAL_BATCH_SIZE = 32
FLAIR_LEARNING_RATE = 1e-4
FINE_TUNE_TRANSFORMER = False

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebooks" / "mult-ml-frameworks" / "flair_feature_family_target_mtl"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flair.device = TORCH_DEVICE

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    horizons=(20, 60),
    min_observations=MIN_ROWS_PER_TASK,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider="fmp",
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        "congress",
        "insider",
        "analyst_rating",
        "price_target",
        "earnings",
    ),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col="high",
    oracle_trade_long_exit_price_col="low",
    oracle_trade_short_entry_price_col="low",
    oracle_trade_short_exit_price_col="high",
)

runtime_config = pd.DataFrame(
    [
        {"setting": "min_market_cap", "value": f"{MIN_MARKET_CAP:,}"},
        {"setting": "date_range", "value": f"{START_DATE} to {END_DATE or 'latest warehouse row'}"},
        {"setting": "smoke_test", "value": SMOKE_TEST},
        {"setting": "smoke_symbol_limit", "value": SMOKE_SYMBOL_LIMIT if SMOKE_TEST else None},
        {"setting": "smoke_target_task_limit", "value": SMOKE_TARGET_TASK_LIMIT if SMOKE_TEST else None},
        {"setting": "smoke_feature_family_limit", "value": SMOKE_FEATURE_FAMILY_LIMIT if SMOKE_TEST else None},
        {"setting": "max_rows_per_task_split", "value": MAX_ROWS_PER_TASK_SPLIT},
        {"setting": "event_families", "value": ", ".join(TARGET_CONFIG.event_families)},
        {"setting": "event_target_rule", "value": "mirrored event-pair rows only; no non-event dates"},
        {"setting": "oracle_target_rule", "value": "oracle buy/sell entry rows only; no non-entry dates as negatives"},
        {"setting": "oracle_trade_k", "value": str(TARGET_CONFIG.oracle_trade_k_by_frequency)},
        {"setting": "text_rule", "value": "feature key-value pairs only; numeric values rounded to 2 decimals; no symbol/date in text"},
        {"setting": "torch_device", "value": str(TORCH_DEVICE)},
        {"setting": "flair_transformer", "value": FLAIR_TRANSFORMER},
    ]
)
display(runtime_config)

,setting,value
0,min_market_cap,"10,000,000,000"
1,date_range,1990-01-01 to latest warehouse row
2,smoke_test,False
3,smoke_symbol_limit,None
4,smoke_target_task_limit,None
5,smoke_feature_family_limit,None
6,max_rows_per_task_split,None
7,event_families,"congress, insider, analyst_rating, price_targe..."
8,event_target_rule,mirrored event-pair rows only; no non-event dates
9,oracle_target_rule,oracle buy/sell entry rows only; no non-entry ...


## Build the Feature/Target Matrix

This mirrors the Quant Warehouse `feature_family_binary_target_matrix` notebook and keeps the `1T+` FMP universe. The output is still tabular at this point; text rows are created in the next section.

In [3]:
started = perf_counter()

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f"Universe source: {universe_source}")
print(f"Eligible symbols before smoke limit: {len(symbols)}")
if SMOKE_TEST and SMOKE_SYMBOL_LIMIT is not None:
    symbols = tuple(symbols[: int(SMOKE_SYMBOL_LIMIT)])
print(f"Symbols used: {len(symbols)}")
print(symbols)

display(eligibility.head(30))

Universe source: catalog:fmp:fallback_after_OpenBBError
Eligible symbols before smoke limit: 828
Symbols used: 828
('A', 'AA', 'AAL', 'AAOI', 'AAON', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEIS', 'AEP', 'AES', 'AFG', 'AFL', 'AFRM', 'AGNC', 'AGR', 'AGX', 'AHR', 'AIG', 'AIT', 'AIZ', 'AIZN', 'AJG', 'AKAM', 'ALAB', 'ALB', 'ALGM', 'ALGN', 'ALL', 'ALLE', 'ALLY', 'ALNY', 'AM', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMH', 'AMKR', 'AMP', 'AMT', 'AMZN', 'ANET', 'ANSS', 'ANTM', 'AON', 'APA', 'APC', 'APD', 'APG', 'APH', 'APLD', 'APO', 'APOS', 'APP', 'APTV', 'AQNB', 'AR', 'ARCC', 'ARES', 'ARMK', 'ARW', 'ARWR', 'ASML', 'ASTS', 'ATI', 'ATO', 'ATVI', 'AUR', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'AXSM', 'AYI', 'AZO', 'AZPN', 'BA', 'BAC', 'BALL', 'BAX', 'BBIO', 'BBY', 'BDX', 'BE', 'BEN', 'BEPH', 'BEPI', 'BF-B', 'BG', 'BGNE', 'BIIB', 'BIPI', 'BJ', 'BK', 'BKDT', 'BKI', 'BKNG', 'BKR', 'BLD', 'BLK', 'BLL', 'BMRN', 'BMY', 'BNH', 'BNJ', 'BNY', '

,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4.819979e+12
1,AAPL,True,ok,4.322489e+12
2,GOOGL,True,ok,4.186397e+12
3,GOOG,True,ok,4.185793e+12
4,MSFT,True,ok,2.777787e+12
5,AMZN,True,ok,2.518345e+12
6,SPCX,True,ok,2.020744e+12
7,AVGO,True,ok,1.808594e+12
8,TSLA,True,ok,1.433220e+12
9,META,True,ok,1.427102e+12


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print(
    {
        "screened_symbols": len(symbols),
        "raw_feature_panel_rows": len(raw_feature_panel),
        "raw_features": raw_feature_metadata["feature"].nunique(),
        **{key: round(value, 3) for key, value in feature_timings.items()},
    }
)

display(feature_diagnostics.sort_values(["status", "rows"], ascending=[True, False]).head(30))

{'screened_symbols': 828, 'raw_feature_panel_rows': 5111257, 'raw_features': 418, 'raw_panel_build_seconds': 88.174}


,symbol,status,rows,features,seconds
663,SOJB,no_features,NaN,NaN,NaN
1,AA,ok,9186.0,376.0,0.086376
5,AAPL,ok,9186.0,376.0,0.090205
10,ABT,ok,9186.0,376.0,0.107010
13,ADBE,ok,9186.0,376.0,0.094502
14,ADI,ok,9186.0,376.0,0.090485
15,ADM,ok,9186.0,376.0,0.091912
16,ADP,ok,9186.0,376.0,0.091411
17,ADSK,ok,9186.0,376.0,0.084180
18,AEE,ok,9186.0,376.0,0.086971


In [ ]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics.loc[event_diagnostics["combined_rows"].gt(0), "symbol"].sort_values().tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel["symbol"].isin(event_symbols)].copy()

selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    raw_feature_metadata,
    max_features=int(raw_feature_metadata["feature"].nunique()),  # no per-family cap; keep compatibility with installed quant-warehouse
)

feature_inventory = (
    selected_feature_metadata
    .groupby(["source", "family"])
    .agg(feature_count=("feature", "nunique"))
    .reset_index()
    .sort_values(["source", "family"])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[["symbol", "date"]],
    events,
    TARGET_CONFIG,
)

oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print(
    {
        "event_symbols": len(event_symbols),
        "event_rows": len(events),
        "selected_features": len(selected_features),
        "feature_families": selected_feature_metadata["family"].nunique(),
        "target_rows": len(target_panel),
        "target_columns": target_metadata["target"].nunique(),
        "event_load_seconds": round(event_load_seconds, 3),
        "oracle_seconds": round(oracle_seconds, 3),
    }
)

display(feature_inventory)
display(target_summary.sort_values(["target_family", "positive_rows"], ascending=[True, False]).head(80))

In [6]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    selected_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=MIN_ROWS_PER_TASK,
    min_positive_rows=MIN_POSITIVE_ROWS_PER_TASK,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
)



PROFILE_FEATURE_SOURCE = FMP_EQUITY_PROFILE_FEATURE_SOURCE
PROFILE_FEATURE_FAMILY = FMP_EQUITY_PROFILE_FEATURE_FAMILY
PROFILE_FEATURE_COLUMNS = list(FMP_EQUITY_PROFILE_FEATURE_COLUMNS)

training_panel, selected_feature_metadata = add_fmp_equity_profile_feature_family(
    training_panel,
    selected_feature_metadata,
    warehouse=WAREHOUSE,
    provider=FEATURE_CONFIG.provider,
)
profile_feature_audit = training_panel[PROFILE_FEATURE_COLUMNS].notna().mean().reset_index(name="coverage").rename(columns={"index": "feature"})
print({"profile_feature_family": PROFILE_FEATURE_FAMILY, "profile_features": PROFILE_FEATURE_COLUMNS})
display(profile_feature_audit)

training_panel, selected_feature_metadata = add_fmp_event_context_feature_families(
    training_panel,
    selected_feature_metadata,
    events,
)
EVENT_CONTEXT_FEATURE_FAMILY_KEYS = {("fmp", family) for family, _ in FMP_EVENT_CONTEXT_FEATURE_FAMILIES.values()}
event_context_feature_inventory = (
    selected_feature_metadata.loc[selected_feature_metadata["family"].isin([family for family, _ in FMP_EVENT_CONTEXT_FEATURE_FAMILIES.values()])]
    .groupby(["source", "family"], as_index=False)
    .agg(feature_count=("feature", "nunique"))
)
print({"event_context_feature_families": sorted(event_context_feature_inventory["family"].tolist())})
display(event_context_feature_inventory)

usable_matrix = matrix.query("status == 'usable'").copy()
if SMOKE_TEST and not usable_matrix.empty:
    if SMOKE_TARGET_TASK_LIMIT is not None:
        keep_targets = (
            usable_matrix
            .sort_values(["positive_rows", "rows"], ascending=[False, False])["target"]
            .drop_duplicates()
            .head(int(SMOKE_TARGET_TASK_LIMIT))
            .tolist()
        )
        usable_matrix = usable_matrix.loc[usable_matrix["target"].isin(keep_targets)].copy()
    if SMOKE_FEATURE_FAMILY_LIMIT is not None:
        keep_families = (
            usable_matrix
            .sort_values(["mean_abs_smd", "positive_rows"], ascending=[False, False])[["source", "feature_family"]]
            .drop_duplicates()
            .head(int(SMOKE_FEATURE_FAMILY_LIMIT))
        )
        usable_matrix = usable_matrix.merge(keep_families, on=["source", "feature_family"], how="inner")
usable_targets = usable_matrix["target"].drop_duplicates().tolist()
usable_feature_families = usable_matrix[["source", "feature_family"]].drop_duplicates()

print(
    {
        "matrix_rows": len(matrix),
        "usable_pairs": len(usable_matrix),
        "training_panel_rows": len(training_panel),
        "usable_targets": len(usable_targets),
        "usable_feature_families": len(usable_feature_families),
        "elapsed_seconds": round(perf_counter() - started, 3),
    }
)

display(usable_matrix.head(80))

{'profile_feature_family': 'fmp_equity_profile', 'profile_features': ['date', 'symbol', 'company_name', 'exchange', 'country', 'sector', 'industry']}


,feature,coverage
0,date,1.0
1,symbol,1.0
2,company_name,1.0
3,exchange,1.0
4,country,1.0
5,sector,1.0
6,industry,1.0


{'matrix_rows': 720, 'usable_pairs': 675, 'training_panel_rows': 5081962, 'usable_targets': 45, 'usable_feature_families': 15, 'elapsed_seconds': 1372.797}


,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
45,financetoolkit,ft_ratios_liquidity,oracle_trade,target_oracle_trade_entry__YE_k12_any,7,5000141,448248,0.089647,1.000000,0.001856,usable
46,financetoolkit,ft_ratios_profitability,oracle_trade,target_oracle_trade_entry__YE_k12_any,14,5000141,448248,0.089647,1.000000,0.001312,usable
47,financetoolkit,ft_ratios_efficiency,oracle_trade,target_oracle_trade_entry__YE_k12_any,5,5000141,448248,0.089647,1.000000,0.001285,usable
48,financetoolkit,ft_ratios_solvency,oracle_trade,target_oracle_trade_entry__YE_k12_any,15,5000141,448248,0.089647,1.000000,0.001183,usable
49,financetoolkit,ft_ratios_valuation,oracle_trade,target_oracle_trade_entry__YE_k12_any,24,5000141,448248,0.089647,0.993319,0.000804,usable
...,...,...,...,...,...,...,...,...,...,...,...
120,financetoolkit,ft_ratios_profitability,oracle_trade,target_oracle_trade_entry__YE_k7_any,14,5000141,282869,0.056572,1.000000,0.002046,usable
121,financetoolkit,ft_ratios_liquidity,oracle_trade,target_oracle_trade_entry__YE_k7_any,7,5000141,282869,0.056572,1.000000,0.001306,usable
122,financetoolkit,ft_ratios_efficiency,oracle_trade,target_oracle_trade_entry__YE_k7_any,5,5000141,282869,0.056572,1.000000,0.001238,usable
123,financetoolkit,ft_ratios_solvency,oracle_trade,target_oracle_trade_entry__YE_k7_any,15,5000141,282869,0.056572,1.000000,0.001056,usable


## Convert Feature Families to Text Rows

Each model example is long-form:

```text
sample = one actual event (date, symbol, feature_family, target_task), labeled by mirrored event side
text   = feature_family=<family> source=<source> feature_a=1.23 feature_b=-4.56 ...
label  = mirrored event label, such as buy/sell, upgrade/downgrade, raise/cut, or beat/miss
```

`symbol` and `date` stay in the dataframe for splitting and auditing, and are also included in the explicit `fmp_equity_profile` context feature family by design.

In [7]:
def _allowed_feature_families(matrix: pd.DataFrame) -> set[tuple[str, str]] | None:
    if not (SMOKE_TEST and SMOKE_FEATURE_FAMILY_LIMIT is not None):
        return None
    allowed_families = (
        matrix.query("status == 'usable'")[["source", "feature_family"]]
        .drop_duplicates()
        .head(int(SMOKE_FEATURE_FAMILY_LIMIT))
    )
    allowed = set(map(tuple, allowed_families.to_numpy()))
    if "PROFILE_FEATURE_SOURCE" in globals() and "PROFILE_FEATURE_FAMILY" in globals():
        allowed.add((PROFILE_FEATURE_SOURCE, PROFILE_FEATURE_FAMILY))
    if "EVENT_CONTEXT_FEATURE_FAMILY_KEYS" in globals():
        allowed.update(EVENT_CONTEXT_FEATURE_FAMILY_KEYS)
    return allowed


def task_specs_from_matrix(matrix: pd.DataFrame) -> list[dict[str, object]]:
    usable = matrix.query("status == 'usable'").copy()
    if usable.empty:
        return []
    available_targets = set(usable["target"].drop_duplicates().astype(str))
    specs = event_pair_task_specs(TARGET_CONFIG, available_targets) + oracle_side_task_specs(available_targets)
    if SMOKE_TEST and SMOKE_TARGET_TASK_LIMIT is not None:
        specs = specs[: int(SMOKE_TARGET_TASK_LIMIT)]
    return specs


def estimate_all_dates_task_grid_rows(
    training_panel: pd.DataFrame,
    feature_metadata: pd.DataFrame,
    matrix: pd.DataFrame,
    *,
    min_feature_coverage: float,
) -> int:
    task_specs = task_specs_from_matrix(matrix)
    if not task_specs:
        return 0

    allowed = _allowed_feature_families(matrix)
    total_family_rows = 0
    for (source, family), family_meta in feature_metadata.groupby(["source", "family"], sort=True):
        if allowed is not None and (str(source), str(family)) not in allowed:
            continue
        family_features = [feature for feature in family_meta["feature"].drop_duplicates().tolist() if feature in training_panel.columns]
        if not family_features:
            continue
        total_family_rows += int(training_panel[family_features].notna().mean(axis=1).ge(min_feature_coverage).sum())
    return total_family_rows * len(task_specs)


def build_flair_long_frame(
    training_panel: pd.DataFrame,
    feature_metadata: pd.DataFrame,
    matrix: pd.DataFrame,
    *,
    min_feature_coverage: float,
    max_rows_per_task_split: int | None = None,
) -> pd.DataFrame:
    usable = matrix.query("status == 'usable'").copy()
    if usable.empty:
        raise RuntimeError("No usable feature-family/target pairs were found for Flair training.")
    task_specs = task_specs_from_matrix(matrix)
    if not task_specs:
        raise RuntimeError("No mirrored event-pair or oracle side tasks are available after filtering.")

    allowed = _allowed_feature_families(matrix)
    result = build_event_feature_text_dataset(
        training_panel,
        feature_metadata,
        task_specs,
        config=EventFeatureDatasetConfig(min_feature_coverage=min_feature_coverage),
        allowed_feature_families=allowed,
        allowed_feature_families_by_task=fmp_event_context_allowed_feature_families_by_task(task_specs, allowed),
    )
    if result.rows.empty:
        raise RuntimeError("Feature-family text conversion produced no event-date training rows.")
    out = result.rows
    if max_rows_per_task_split is not None:
        out = cap_rows_per_task_split(out, max_rows_per_task_split)
    return out


def chronological_split(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    ordered = frame.sort_values(["date", "symbol", "feature_family", "target_task"]).reset_index(drop=True)
    splits = {
        "train": ordered.loc[ordered["date"].le(TRAIN_END)].copy(),
        "dev": ordered.loc[ordered["date"].gt(TRAIN_END) & ordered["date"].le(DEV_END)].copy(),
        "test": ordered.loc[ordered["date"].gt(DEV_END)].copy(),
    }
    if min(len(split) for split in splits.values()) > 0:
        return splits

    first = max(2, int(len(ordered) * 0.70))
    second = max(first + 1, int(len(ordered) * 0.85))
    return {
        "train": ordered.iloc[:first].copy(),
        "dev": ordered.iloc[first:second].copy(),
        "test": ordered.iloc[second:].copy(),
    }


def cap_rows_per_task_split(frame: pd.DataFrame, max_rows: int) -> pd.DataFrame:
    if max_rows <= 0:
        return frame
    group_columns = ["split", "task_id"] if "split" in frame.columns else ["task_id"]
    pieces = []
    for _, group in frame.groupby(group_columns, sort=False):
        if len(group) <= max_rows:
            pieces.append(group)
            continue
        labels = group["label"].astype(str)
        per_label_budget = max(1, max_rows // max(1, labels.nunique()))
        sampled_parts = []
        for _, label_group in group.groupby("label", sort=False):
            sampled_parts.append(label_group.sample(n=min(len(label_group), per_label_budget), random_state=RANDOM_SEED))
        sampled = pd.concat(sampled_parts, axis=0)
        if len(sampled) < max_rows:
            remainder = group.drop(index=sampled.index)
            if not remainder.empty:
                sampled = pd.concat(
                    [sampled, remainder.sample(n=min(len(remainder), max_rows - len(sampled)), random_state=RANDOM_SEED)],
                    axis=0,
                )
        pieces.append(sampled)
    return (
        pd.concat(pieces, ignore_index=True)
        .sort_values(["date", "symbol", "feature_family", "target_task"])
        .reset_index(drop=True)
    )

In [ ]:
all_dates_task_grid_rows = estimate_all_dates_task_grid_rows(
    training_panel,
    selected_feature_metadata,
    usable_matrix,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
)

long_frame = build_flair_long_frame(
    training_panel,
    selected_feature_metadata,
    usable_matrix,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
    max_rows_per_task_split=None,  # cap after chronological split so each split/task is represented
)


print(
    {
        "smoke_test": SMOKE_TEST,
        "all_dates_task_grid_rows_before_event_filter": all_dates_task_grid_rows,
        "event_only_rows_before_cap": len(long_frame),
        "rows_removed_by_event_only_rule": all_dates_task_grid_rows - len(long_frame),
        "row_reduction_pct": round(100 * (1 - len(long_frame) / all_dates_task_grid_rows), 2) if all_dates_task_grid_rows else None,
        "max_rows_per_task_split": MAX_ROWS_PER_TASK_SPLIT,
    }
)

splits = chronological_split(long_frame)
for split_name, split_frame in splits.items():
    split_frame["split"] = split_name
long_frame = pd.concat(splits.values(), ignore_index=True)
if MAX_ROWS_PER_TASK_SPLIT is not None:
    long_frame = cap_rows_per_task_split(long_frame, MAX_ROWS_PER_TASK_SPLIT)
    splits = {name: frame.drop(columns=["split"], errors="ignore") for name, frame in long_frame.groupby("split", sort=False)}
else:
    splits = {name: frame.drop(columns=["split"], errors="ignore") for name, frame in long_frame.groupby("split", sort=False)}

long_frame["token_estimate"] = long_frame["text"].str.split().str.len()
max_token_estimate = int(long_frame["token_estimate"].max()) if not long_frame.empty else 0

text_audit = (
    long_frame
    .groupby(["split", "task_id"])
    .agg(
        rows=("label", "size"),
        labels=("label", lambda s: dict(s.value_counts())),
        feature_families=("feature_family", "nunique"),
        median_token_estimate=("token_estimate", "median"),
        max_token_estimate=("token_estimate", "max"),
    )
    .reset_index()
    .sort_values(["task_id", "split"])
)

print(
    {
        "all_dates_task_grid_rows_before_event_filter": all_dates_task_grid_rows,
        "event_rows": len(long_frame),
        "rows_removed_by_event_only_rule": all_dates_task_grid_rows - len(long_frame),
        "tasks": long_frame["task_id"].nunique(),
        "feature_families": long_frame["feature_family"].nunique(),
        "symbols": long_frame["symbol"].nunique(),
        "min_date": long_frame["date"].min(),
        "max_date": long_frame["date"].max(),
    }
)

display(text_audit.head(120))
display(long_frame[["date", "symbol", "feature_family", "target_task", "split", "source", "task_id", "label", "positive_target_col", "negative_target_col", "token_estimate", "text"]].head(10))

assert {"positive_target_col", "negative_target_col"}.issubset(long_frame.columns), "target pair lineage is required"
assert not long_frame["target_task"].astype(str).str.endswith("_any").any(), "oracle any targets must not be used as binary no-event tasks"
assert long_frame.loc[long_frame["target_task"].astype(str).str.contains("oracle"), "target_task"].nunique() <= 1, "oracle trades must be one buy/sell task across all k values"
assert not long_frame["label"].astype(str).isin(["positive", "negative"]).any(), "labels must be mirrored event names, not generic positive/negative no-event labels"

_pair_values = training_panel.set_index(["symbol", "date"])
_audit = long_frame[["symbol", "date", "positive_target_col", "negative_target_col", "label"]].copy()
_audit["_row_id"] = np.arange(len(_audit))
_positive_lookup = (
    _audit.assign(target=_audit["positive_target_col"].astype(str).str.split("|"))
    .explode("target")[["_row_id", "symbol", "date", "target"]]
    .assign(side="positive")
)
_negative_lookup = (
    _audit.assign(target=_audit["negative_target_col"].astype(str).str.split("|"))
    .explode("target")[["_row_id", "symbol", "date", "target"]]
    .assign(side="negative")
)
_lookup = pd.concat([_positive_lookup, _negative_lookup], ignore_index=True)
_target_long = _pair_values.reset_index().melt(
    id_vars=["symbol", "date"],
    var_name="target",
    value_name="value",
)
_lookup = _lookup.merge(_target_long, on=["symbol", "date", "target"], how="left")
_lookup["value"] = pd.to_numeric(_lookup["value"], errors="coerce").fillna(0).astype("int8")
_side_values = _lookup.groupby(["_row_id", "side"])["value"].max().unstack(fill_value=0).astype("int8")
assert (_side_values["positive"] + _side_values["negative"]).eq(1).all(), "event/oracle rows must be exact buy/sell or mirrored event dates with exactly one side; no-event dates are forbidden"
assert not ((_side_values["positive"].eq(1)) & long_frame["label"].astype(str).eq("negative")).any(), "positive event side was mislabeled"
assert not ((_side_values["negative"].eq(1)) & long_frame["label"].astype(str).eq("positive")).any(), "negative event side was mislabeled"


## Build a Flair Native Multitask Corpus

Each target column is a separate Flair task. The same feature-family text can appear under multiple tasks, but each row carries exactly one `multitask_id` and one mirrored event label for that target.

In [ ]:
class LazyMultitaskDataset(FlairDataset):
    """Create Flair Sentence objects on demand instead of materializing the full split."""

    def __init__(self, frame: pd.DataFrame):
        required = ["text", "label_type", "label", "task_id"]
        missing = [column for column in required if column not in frame.columns]
        if missing:
            raise ValueError(f"Missing lazy dataset columns: {missing}")
        self.texts = frame["text"].astype(str).to_numpy(copy=True)
        self.label_types = frame["label_type"].astype(str).to_numpy(copy=True)
        self.labels = frame["label"].astype(str).to_numpy(copy=True)
        self.task_ids = frame["task_id"].astype(str).to_numpy(copy=True)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> Sentence:
        sentence = Sentence(self.texts[index])
        sentence.add_label(self.label_types[index], self.labels[index])
        sentence.add_label("multitask_id", self.task_ids[index])
        return sentence

    def is_in_memory(self) -> bool:
        return False


def build_multitask_corpus(splits: dict[str, pd.DataFrame]) -> Corpus:
    return Corpus(
        train=LazyMultitaskDataset(splits["train"]),
        dev=LazyMultitaskDataset(splits["dev"]),
        test=LazyMultitaskDataset(splits["test"]),
        sample_missing_splits=False,
    )


def label_dictionary_for_task(frame: pd.DataFrame, task_id: str) -> Dictionary:
    dictionary = Dictionary(add_unk=False)
    labels = frame.loc[frame["task_id"].eq(task_id), "label"].astype(str).drop_duplicates().sort_values()
    for label in labels:
        dictionary.add_item(label)
    return dictionary


def build_binary_multitask_model(corpus: Corpus, task_ids: list[str]) -> MultitaskModel:
    embeddings = TransformerDocumentEmbeddings(
        FLAIR_TRANSFORMER,
        fine_tune=FINE_TUNE_TRANSFORMER,
        layers="-1",
        layer_mean=False,
        allow_long_sentences=False,
        transformers_tokenizer_kwargs={"model_max_length": 512},
    )
    classifiers = []
    for task_id in task_ids:
        classifier = TextClassifier(
            embeddings,
            label_type=task_id,
            label_dictionary=label_dictionary_for_task(long_frame, task_id),
        )
        classifiers.append(classifier)
    model = MultitaskModel(
        classifiers,
        task_ids=task_ids,
        loss_factors=[1.0 for _ in task_ids],
        use_all_tasks=False,
    )
    return model.to(flair.device)


def predict_task_classifier(model: MultitaskModel, task_id: str, rows: pd.DataFrame) -> np.ndarray:
    task = model.tasks[task_id]
    sentences = [Sentence(str(text)) for text in rows["text"].tolist()]
    task.predict(sentences, label_name="pred", mini_batch_size=FLAIR_EVAL_BATCH_SIZE)
    return np.array([sentence.get_labels("pred")[0].value for sentence in sentences])


def evaluate_multitask_model(model: MultitaskModel, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task_id, task_rows in frame.groupby("task_id", sort=True):
        y_true = task_rows["label"].astype(str).to_numpy()
        if len(np.unique(y_true)) < 2:
            rows.append(
                {
                    "task_id": task_id,
                    "rows": len(task_rows),
                    "labels": dict(task_rows["label"].astype(str).value_counts()),
                    "accuracy": np.nan,
                    "macro_f1": np.nan,
                    "macro_precision": np.nan,
                    "macro_recall": np.nan,
                    "note": "single_class_test_set",
                }
            )
            continue
        y_pred = predict_task_classifier(model, task_id, task_rows)
        rows.append(
            {
                "task_id": task_id,
                "target_task": task_rows["target_task"].iloc[0],
                "rows": len(task_rows),
                "labels": dict(task_rows["label"].astype(str).value_counts()),
                "accuracy": accuracy_score(y_true, y_pred),
                "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
                "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
                "note": "ok",
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["macro_f1", "rows"], ascending=[False, False], na_position="last").reset_index(drop=True)

In [ ]:
# Execute this cell when you are ready to train.
# It is intentionally separate from dataset construction so you can inspect `long_frame` first.

train_started = perf_counter()
corpus_started = perf_counter()
corpus = build_multitask_corpus(splits)
corpus_seconds = perf_counter() - corpus_started
task_ids = sorted(long_frame["task_id"].drop_duplicates().tolist())
model = build_binary_multitask_model(corpus, task_ids)
trainer = ModelTrainer(model, corpus)

train_result = trainer.fine_tune(
    base_path=ARTIFACT_DIR,
    learning_rate=FLAIR_LEARNING_RATE,
    mini_batch_size=FLAIR_BATCH_SIZE,
    mini_batch_chunk_size=FLAIR_BATCH_CHUNK_SIZE,
    eval_batch_size=FLAIR_EVAL_BATCH_SIZE,
    max_epochs=FLAIR_EPOCHS,
    embeddings_storage_mode="none",
    save_final_model=False,
    create_file_logs=True,
    create_loss_file=True,
)

train_seconds = perf_counter() - train_started
print({"tasks": len(task_ids), "corpus_seconds": round(corpus_seconds, 3), "train_seconds": round(train_seconds, 3), "artifact_dir": str(ARTIFACT_DIR)})

In [ ]:
# Execute after training to inspect task-level results.

test_results = evaluate_multitask_model(model, splits["test"])
display(test_results)

analysis_lines = [
    "## Training Notes",
    "",
    f"- Universe: {len(symbols)} symbols from FMP with market cap >= ${MIN_MARKET_CAP:,.0f}.",
    f"- Long-form event rows: {len(long_frame):,} across {long_frame['feature_family'].nunique()} feature families and {long_frame['task_id'].nunique()} target tasks.",
    f"- Event-only filtering removed {all_dates_task_grid_rows - len(long_frame):,} rows versus the all-dates task grid ({all_dates_task_grid_rows:,} candidate rows before filtering).",
    "- Text excludes symbol and date by assertion; they are metadata only for splitting and auditing.",
    "- Numeric feature values are rounded to two decimals before string formatting.",
    "- Feature-family text includes every available selected feature for the family; rows, not columns, were the dataset-size bottleneck.",
    "- `fmp_equity_profile` is a categorical context/profile family with date, symbol, company_name, exchange, country, sector, and industry.",
    "- FMP event-pair tasks include only actual mirrored event dates. Non-event dates are excluded and are never used as negative labels.",
    "- Oracle uses one buy/sell task across all configured k values. Only oracle buy and sell entry dates are included; non-entry dates are excluded and are never used as negative labels.",
]
display(Markdown("\n".join(analysis_lines)))